In [1]:
from tornado import gen

csv_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.csv"
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cbd"


In [2]:
import re
import pandas as pd
from pathlib import Path

def parse_cdb(cdb_path):
    """Parse NLSY79 codebook (.cdb) - simple version"""
    codebook = {}
    text = Path(cdb_path).read_text()

    # Split by long dashes
    blocks = re.split(r"-{40,}", text)

    for block in blocks:
        # Match header: H00016.00    [H40-BPAR-1]    Survey Year: XRND
        m = re.match(r"\s*(\w+\.\d+)\s+\[([^\]]+)\]\s+Survey Year:\s*(\S+)", block)
        if not m:
            continue

        rnum = m.group(1).replace(".", "")  # H0001600
        qname = m.group(2)                   # H40-BPAR-1
        year = m.group(3)                    # XRND

        # Extract question (after "PRIMARY VARIABLE")
        question = ""
        qm = re.search(r"PRIMARY VARIABLE\s*\n\s*\n\s*(.+?)(?:\n\s*\n|$)", block, re.DOTALL)
        if qm:
            question = qm.group(1).strip().replace("\n", " ")

        codebook[rnum] = {
            "rnum": rnum,
            "qname": qname,
            "year": year,
            "question": question,
        }

    return codebook


In [3]:
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cdb"
codebook = parse_cdb(cdb_path)

cb_df = pd.DataFrame(codebook.values())


In [4]:
cb_df["qname"].value_counts()

qname
Q9-89.01    13
Q9-91.01    13
Q9-89.02    13
Q9-91.02    13
Q9-88.01    10
            ..
Q9-90.04     1
Q9-91.07     1
Q9-91.08     1
Q9-89.10     1
Q9-91.10     1
Name: count, Length: 321, dtype: int64

In [5]:
invalid_year = ~cb_df["year"].astype(str).str.fullmatch(r"\d{4}")

unique_qname = cb_df.groupby("qname")["qname"].transform("size") == 1

# retroperspecitv questions
living_struc = (
    cb_df["year"].astype(str).eq("1988")
    & cb_df["question"]
        .astype(str)
        .str.lower()
        .str.startswith(("lived with", "lived in"), na=False)
)

##sind die ganzen 1988 struktur variablen auch drinnen -> wie identifizieren wir die? questinon text ist mit lived with..
cbd_p = cb_df[invalid_year | unique_qname & ~living_struc]
cbd_pl = cb_df[~(invalid_year | unique_qname) | (cb_df["rnum"] == "R0000100")]
cbd_ls = cb_df[living_struc | (cb_df["rnum"] == "R0000100")]

In [6]:
print(len(cbd_p))
print(len(cbd_pl))
print(len(cbd_ls))


31
181
255


In [7]:
df_csv = pd.read_csv(csv_path)
csv_pl = df_csv[cbd_pl["rnum"]]
csv_p = df_csv[cbd_p["rnum"]]
csv_ls = df_csv[cbd_ls["rnum"]]

In [10]:
#variables we want to make to long
rnum_vars_p = cbd_p["rnum"].tolist()
print(len(rnum_vars_p))
# melt: von wide zu long. id_vars ist  'R0000100'
melted_p = csv_p.melt(id_vars=["R0000100"], value_vars=rnum_vars_p,
                     var_name="rnum", value_name="value")


melted_p = melted_p.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_p = melted_p.rename(columns={"R0000100": "caseid"})
melted_p = melted_p.sort_values(["caseid", "rnum"]).reset_index(drop=True)

# 5) Erzeuge eine numerische survey-year Spalte (syear) wenn year eine vierstellige Jahreszahl ist
melted_p["syear"] = pd.to_numeric(melted_p["year"], errors="coerce")


31


In [34]:
pivoted_p = melted_p.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [35]:
pivoted_p["birthyear"] =  1979 - pivoted_p["FAM-1B"]

In [25]:
#variables we want to make to long
rnum_vars = cbd_pl["rnum"].tolist()
print(len(rnum_vars))
# melt: von wide zu long. id_vars ist  'R0000100'
melted = csv_pl.melt(id_vars=["R0000100"], value_vars=rnum_vars,
                     var_name="rnum", value_name="value")

# 3) Um aussagekräftige Meta-Information (qname, year, question) beizufügen, mergen wir mit cb_df
melted = melted.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")
# Erklärung:
# - cb_df enthält die Informationen, die parse_cdb erzeugt hat
# - nach dem Merge hat jede Zeile zusätzlich qname, year und question

melted = melted.rename(columns={"R0000100": "caseid"})
melted = melted.sort_values(["caseid", "rnum"]).reset_index(drop=True)

# 5) Erzeuge eine numerische survey-year Spalte (syear) wenn year eine vierstellige Jahreszahl ist
melted["syear"] = pd.to_numeric(melted["year"], errors="coerce")


181


In [26]:
pivoted_pl = melted.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()


In [32]:
csv_p["birthyear"] =  1979 - csv_p["R0000600"]


In [22]:
pivoted_ls = melted_ls.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [28]:
rnum_vars_ls = cbd_ls["rnum"].tolist()

# melt: von wide zu long. id_vars ist  'R0000100'
melted_ls = csv_ls.melt(id_vars=["R0000100"], value_vars=rnum_vars_ls,
                     var_name="rnum", value_name="value")

melted_ls = melted_ls.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_ls = melted_ls.rename(columns={"R0000100": "caseid"})
melted_ls = melted_ls.sort_values(["caseid", "rnum"]).reset_index(drop=True)

melted_ls["syear"] = pd.to_numeric(melted_ls["year"], errors="coerce")

In [29]:
def reshape_retrospective(df, var_prefix):
    df = df.dropna(subset=['birthyear'])

    cols = []
    for col in df.columns:
        if col.startswith(var_prefix.upper()):
            # Extrahiere die Zahl am Ende
            match = re.search(r'(\d+)$', col)
            if match:
                age_num = int(match.group(1))
                if 0 <= age_num <= 18:
                    cols.append(col)
    if not cols:
        print(f"Keine passenden Spalten gefunden!")
        return pd.DataFrame()

    long = df.melt(
        id_vars=['caseid', 'year', 'birthyear'],
        value_vars=cols,
        var_name='suffix',
        value_name=var_prefix
    )

    long['age'] = long['suffix'].str.extract(r'(\d+)$').astype(int)
    long['year'] = long['birthyear'] + long['age']  # ✓ direkt als 'year'

    return long[['caseid', 'year', 'age', 'birthyear', var_prefix]].sort_values(['caseid', 'year']).reset_index(drop=True)

In [36]:
# Merge mit caseid  year
pivoted_ls_with_birth = pivoted_ls.merge(
    pivoted_p[['caseid', 'birthyear']],  # Nur eindeutige caseid-birthyear
    on='caseid',
    how='left'
)
# Anwenden
adopt_long = reshape_retrospective(pivoted_ls_with_birth, 'adoptd')
pivoted_ls_with_birth

qname,caseid,year,ADOPTD0,ADOPTD1,ADOPTD10,ADOPTD11,ADOPTD12,ADOPTD13,ADOPTD14,ADOPTD15,...,STPMOM2,STPMOM3,STPMOM4,STPMOM5,STPMOM6,STPMOM7,STPMOM8,STPMOM9,STPMOM96,birthyear
0,1,1988,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,1959.0
1,1,1988,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,NaN
2,1,1988,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,NaN
3,1,1988,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,NaN
4,1,1988,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114169,12686,1988,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,...,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,1.0,NaN
114170,12686,1988,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,...,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,1.0,NaN
114171,12686,1988,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,...,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,1.0,NaN
114172,12686,1988,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,...,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,-4.0,1.0,NaN


In [37]:
cols_ending_18 = [col for col in pivoted_ls_with_birth.columns
                  if re.search(r'18$', col)]
prefixes = [re.sub(r'\d+$', '', col) for col in cols_ending_18]

long_dfs = [reshape_retrospective(pivoted_ls_with_birth, prefix) for prefix in prefixes]

result_ls = pd.concat(long_dfs, axis=1)
result_ls = result_ls.loc[:, ~result_ls.columns.duplicated()].sort_values(['caseid', 'year']).reset_index(drop=True)


In [38]:

merged_1 = result_ls.merge(pivoted_pl, on=['caseid', 'year'], how='outer')

merged_1['birthyear'] = merged_1.groupby('caseid')['birthyear'].transform(
    lambda x: x.fillna(x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

result_ls['year'] = result_ls['year'].astype(str)
pivoted_pl['year'] = pivoted_pl['year'].astype(str)
pivoted_p['year'] = pivoted_p['year'].astype(str)

# Berechne age
merged_1['age'] = pd.to_numeric(merged_1['year'], errors='coerce') - merged_1['birthyear'].astype(float)

# Merge mit pivoted_p
final_result = merged_1.merge(pivoted_p, on=['caseid', 'year'], how='outer')
final_result = final_result.sort_values(['caseid', 'year']).reset_index(drop=True)



ValueError: You are trying to merge on float64 and str columns for key 'year'. If you wish to proceed you should use pd.concat

In [ ]:
final_result